In [0]:
df_crm_cust = spark.sql("""
SELECT * FROM sqldatawarehouse.bronze.crm_customers
""")

In [0]:
from pyspark.sql import functions as F

# Incremental processing: Only process data not yet in silver
target_table = "sqldatawarehouse.silver.crm_customers"

try:
    # Check if silver table exists and get max process_timestamp
    max_process_ts = spark.sql(f"""
        SELECT COALESCE(MAX(process_timestamp), CAST('1900-01-01' AS TIMESTAMP)) as max_ts
        FROM {target_table}
    """).collect()[0]['max_ts']
    
    print(f"📊 Incremental mode: Processing data ingested after {max_process_ts}")
    
    # Filter bronze data to only include records newer than max process_timestamp
    initial_count = df_crm_cust.count()
    df_crm_cust = df_crm_cust.filter(F.col("ingestion_time") > F.lit(max_process_ts))
    filtered_count = df_crm_cust.count()
    
    print(f"   Initial records from bronze: {initial_count}")
    print(f"   New records to process: {filtered_count}")
    print(f"   Skipped (already processed): {initial_count - filtered_count}")
    
except Exception as e:
    # Table doesn't exist yet - process all data (first run)
    print(f"📊 Full load mode: Silver table does not exist yet, processing all bronze data")
    print(f"   Records to process: {df_crm_cust.count()}")

### Add metadata columns

### `Check` for Nulls or Duplicates in Primary Key

#### Expectation: No Result

In [0]:
from pyspark.sql import functions as F

result = (
    df_crm_cust.groupBy("cst_id")
      .count()
      .filter((F.col("count") > 1) | F.col("cst_id").isNull())
)

result.show()

### Clear Null and Duplicates

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_cust = (
    df_crm_cust.filter(
        F.col("cst_id").isNotNull() &
        (F.trim(F.col("cst_id")) != "")
    )
)

w = Window.partitionBy("cst_id").orderBy(F.desc("cst_create_date"))

df_crm_cust = (
    df_crm_cust
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") == 1)
    .drop("rank")
)




### Check for unwanted spaces in string values

In [0]:
df = df_crm_cust.where(F.col('cst_lastname') != F.trim(F.col("cst_lastname")))
display(df)

In [0]:
df_crm_cust = df_crm_cust.withColumn('cst_firstname', F.trim(F.col('cst_firstname')))
df_crm_cust = df_crm_cust.withColumn('cst_lastname', F.trim(F.col('cst_lastname')))


### Data Standardization & consistency

In [0]:
display(df_crm_cust.select('cst_gndr').distinct())

In [0]:
df_crm_cust = df_crm_cust.withColumn(
    "cst_gndr",
    F.when(F.trim(F.upper(F.col("cst_gndr"))).isNull(), "Unknown")
     .when(F.trim(F.upper(F.col("cst_gndr"))) == "M", "Male")
     .when(F.trim(F.upper(F.col("cst_gndr"))) == "F", "Female")
     .otherwise("Unknown")
)
display(df_crm_cust.select("cst_gndr").distinct())

In [0]:
display(df_crm_cust.select('cst_marital_status').distinct())

In [0]:
df_crm_cust = df_crm_cust.withColumn(
    "cst_marital_status",
    F.when(F.trim(F.upper(F.col("cst_marital_status"))).isNull(), "Unknown")
     .when(F.trim(F.upper(F.col("cst_marital_status"))) == "M", "Married")
     .when(F.trim(F.upper(F.col("cst_marital_status"))) == "S", "Single")
     .otherwise("Unknown")
)
display(df_crm_cust.select("cst_marital_status").distinct())

In [0]:
import uuid
from datetime import datetime
from pyspark.sql.functions import lit, current_timestamp

# Configuration
SILVER_DB = "sqldatawarehouse.silver"
AUDIT_TABLE = f"{SILVER_DB}.transformation_audit"

# Get batch_id from parameter or generate new one (for standalone runs)
try:
    batch_id = dbutils.widgets.get("batch_id")
    print(f"⚙ Using shared batch_id from master pipeline: {batch_id}")
except:
    batch_id = str(uuid.uuid4())
    print(f"⚙ Running standalone - generated new batch_id: {batch_id}")

In [0]:
# Define audit function only if not already defined by master notebook
if 'log_silver_audit' not in dir():
    def log_silver_audit(source_table, target_table, row_count, transformations_applied, status):
        """
        Log silver layer transformation to audit table
        
        Args:
            source_table: Bronze table(s) used as source
            target_table: Silver table written to
            row_count: Number of rows in the result
            transformations_applied: Description of transformations
            status: SUCCESS or FAILED
        """
        audit_df = spark.createDataFrame([(
            source_table,
            target_table,
            row_count,
            batch_id,
            transformations_applied,
            status,
            datetime.now()
        )], [
            "source_table",
            "target_table",
            "row_count",
            "batch_id",
            "transformations_applied",
            "status",
            "transformation_time"
        ])
        
        audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)
    print("✓ Audit function defined")
else:
    print("✓ Using shared audit function from master pipeline")

In [0]:
# Define source and target
source_table = "sqldatawarehouse.bronze.crm_customers"
target_table = f"{SILVER_DB}.crm_customers"

# List of transformations applied
transformations = [
    "Incremental processing: Only processed records where bronze ingestion_time > max silver process_timestamp",
    "Removed null/empty cst_id",
    "Deduplicated by cst_id (keeping latest by cst_create_date)",
    "Trimmed whitespace from cst_firstname, cst_lastname",
    "Standardized cst_gndr to Male/Female/Unknown",
    "Standardized cst_marital_status to Married/Single/Unknown",
    "Added process_timestamp column to track silver layer load time"
]

try:
    # Update batch_id and add process_timestamp for silver layer
    from pyspark.sql import functions as F
    df_crm_cust = df_crm_cust.withColumns({
        'batch_id': F.lit(batch_id),
        'process_timestamp': F.current_timestamp()
    })
    
    # Get row count
    row_count = df_crm_cust.count()
    
    # Write to silver layer
    df_crm_cust.write \
        .format("delta") \
        .mode("append") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)
    
    print(f"✓ Successfully wrote {row_count} rows to {target_table}")
    
    # Log audit
    log_silver_audit(
        source_table=source_table,
        target_table=target_table,
        row_count=row_count,
        transformations_applied="; ".join(transformations),
        status="SUCCESS"
    )
    
    print(f"✓ Audit logged to {AUDIT_TABLE}")
    
except Exception as e:
    print(f"✗ Failed to write to {target_table}: {str(e)}")
    
    # Log failure
    log_silver_audit(
        source_table=source_table,
        target_table=target_table,
        row_count=0,
        transformations_applied="; ".join(transformations),
        status=f"FAILED: {str(e)[:200]}"
    )
    
    raise

In [0]:
# # Verify the table was created
# print(f"\nVerifying {target_table}:")
# spark.sql(f"DESCRIBE TABLE {target_table}").show(truncate=False)

# print(f"\nRow count in {target_table}: {spark.table(target_table).count()}")

# # Show recent audit logs
# print(f"\nRecent audit logs from {AUDIT_TABLE}:")
# spark.sql(f"""
#     SELECT 
#         target_table,
#         row_count,
#         status,
#         transformation_time,
#         transformations_applied
#     FROM {AUDIT_TABLE}
#     ORDER BY transformation_time DESC
#     LIMIT 5
# """).display(truncate=False)